# MLB Edge Finder — Exploration

Interactive walkthrough of the pipeline stages.

In [1]:
import logging
from datetime import date

from mlb_edge_finder import config

from pybaseball import cache
cache.enable()

config.setup_logging(level=logging.INFO)

## 1. Fetch Odds

In [2]:
from mlb_edge_finder import odds_ingestion

game_date = date.today()
odds_df = odds_ingestion.fetch_odds(game_date, force=True, debug=True)
odds_df.head()

2026-05-12 22:32:08,946 | INFO | mlb_edge_finder.odds_ingestion | Raw API response: 15 game(s) returned
2026-05-12 22:32:08,950 | INFO | mlb_edge_finder.odds_ingestion |   game_id=21ff18fdfd96a55a86867197e834497e  home=Baltimore Orioles  away=New York Yankees  commence_time=2026-05-13T17:06:00Z  local_date=2026-05-13
2026-05-12 22:32:08,951 | INFO | mlb_edge_finder.odds_ingestion |   game_id=67da7d89e932aa7075b0600f26224572  home=Cleveland Guardians  away=Los Angeles Angels  commence_time=2026-05-13T17:11:00Z  local_date=2026-05-13
2026-05-12 22:32:08,951 | INFO | mlb_edge_finder.odds_ingestion |   game_id=4546828df054a4ac1c7634c94b4c7012  home=Cincinnati Reds  away=Washington Nationals  commence_time=2026-05-13T22:41:00Z  local_date=2026-05-13
2026-05-12 22:32:08,952 | INFO | mlb_edge_finder.odds_ingestion |   game_id=198e79fae18be9cb416f60f6da4d7996  home=Pittsburgh Pirates  away=Colorado Rockies  commence_time=2026-05-13T22:41:00Z  local_date=2026-05-13
2026-05-12 22:32:08,953 | INF

,game_id,home_team,away_team,home_odds_american,away_odds_american,commence_time


## 2. Fetch Stats

In [3]:
from mlb_edge_finder import stats_ingestion

stats_df = stats_ingestion.fetch_stats(game_date)
stats_df.head()

,team_abbr,bat_avg,obp,slg,ops,runs_per_game,era,whip,k_per_9,bb_per_9,fip_computed,data_source
0,ATL,0.272,0.335,0.452,0.787,5.547619,3.11,1.15,8.88,3.55,3.822922,mlb_api
1,LAD,0.263,0.342,0.430,0.772,4.952381,3.48,1.14,9.10,2.92,3.506757,mlb_api
2,TB,0.258,0.330,0.379,0.709,4.536585,3.48,1.15,7.90,3.07,3.897765,mlb_api
3,HOU,0.255,0.330,0.421,0.751,4.627907,5.61,1.59,8.97,5.20,4.866875,mlb_api
4,PIT,0.250,0.337,0.390,0.727,4.976190,3.67,1.22,8.89,3.55,3.445657,mlb_api


## 3. Build Features

In [4]:
from mlb_edge_finder import features
from mlb_edge_finder.pitcher_ingestion import fetch_pitcher_stats

# Fetch pitcher stats for today's season (cached after first run)
# Required before build_features() — fetches all pitcher season stats
fetch_pitcher_stats(game_date)

features_df = features.build_features(game_date)
features_df.head()
features_df[features_df.columns[:10]].head()


2026-05-12 22:32:09,432 | INFO | mlb_edge_finder.features | Wrote 0 rows to /Users/jaydengould/Documents/projects/mlb-edge-finder/data/processed/features_2026-05-12.csv


,game_id,home_team,away_team,home_odds_american,away_odds_american,commence_time,home_abbr,away_abbr,home_bat_avg,home_obp


## 4a. Historical Ingestion

Fetch completed regular season results for each training season via `statsapi`.

In [5]:
from mlb_edge_finder import historical_ingestion

# Fetch (or load cached) results for a single season
hist_2024 = historical_ingestion.fetch_historical(2024)
print(f"{len(hist_2024)} games")
hist_2024.head()

2428 games


,game_date,home_name,away_name,home_score,away_score,home_win
0,2024-03-20,San Diego Padres,Los Angeles Dodgers,2,5,0
1,2024-03-21,Los Angeles Dodgers,San Diego Padres,11,15,0
2,2024-03-28,Baltimore Orioles,Los Angeles Angels,11,3,1
3,2024-03-28,Cincinnati Reds,Washington Nationals,8,2,1
4,2024-03-28,San Diego Padres,San Francisco Giants,6,4,1


In [6]:
# Concatenate all training seasons (2023, 2024, 2025)
all_hist = historical_ingestion.fetch_all_historical()
print(f"{len(all_hist)} total games")
all_hist.groupby(all_hist['game_date'].str[:4])['home_win'].agg(['count', 'mean'])

2026-05-12 22:32:09,479 | INFO | mlb_edge_finder.historical_ingestion | fetch_all_historical: 14429 total games across seasons [2019, 2021, 2022, 2023, 2024, 2025]


14429 total games


,count,mean
game_date,,
2019,2423,0.528683
2021,2386,0.538558
2022,2347,0.528334
2023,2418,0.520678
2024,2428,0.521005
2025,2427,0.543881


## 4b. Training Data

Join end-of-season team stats + rolling window stats (last 15 games: runs scored, 
runs allowed, win %, run diff) to each game row to produce the model training set.

Run with `force=True` to rebuild the cache with rolling columns included.

In [7]:
from mlb_edge_finder import training_data

seasons = [2019, 2021, 2022, 2023, 2024, 2025]
# force=True rebuilds cache to include all 6 seasons (2019, 2021-2025; 2020 skipped — 60-game anomaly)
training_df = training_data.build_training_set(seasons, force=True)
print(f"{len(training_df)} rows, {len(training_df.columns)} columns")
training_df.head()


2026-05-12 22:32:09,934 | INFO | mlb_edge_finder.training_data | Wrote 15050 rows (6 seasons) to /Users/jaydengould/Documents/projects/mlb-edge-finder/data/processed/training_2019-2025.csv


15050 rows, 53 columns


,game_date,home_name,away_name,home_score,away_score,home_win,home_starter_name,away_starter_name,home_abbr,away_abbr,...,home_sp_bb_per_9,home_sp_ip,home_sp_fip_computed,away_pitcher_id,away_sp_era,away_sp_whip,away_sp_k_per_9,away_sp_bb_per_9,away_sp_ip,away_sp_fip_computed
0,2019-03-20,Oakland Athletics,Seattle Mariners,7,9,0,Mike Fiers,Marco Gonzales,ATH,SEA,...,2.58,184.2,4.762378,594835.0,3.99,1.31,6.52,2.48,203.0,4.002217
1,2019-03-21,Oakland Athletics,Seattle Mariners,4,5,0,Marco Estrada,Yusei Kikuchi,ATH,SEA,...,3.04,23.2,7.158621,579328.0,5.46,1.52,6.46,2.78,161.2,5.544541
2,2019-03-28,Washington Nationals,New York Mets,0,2,0,Max Scherzer,Jacob deGrom,WSH,NYM,...,1.72,172.1,2.260982,594798.0,2.43,0.97,11.25,1.94,204.0,2.507843
3,2019-03-28,New York Yankees,Baltimore Orioles,7,2,1,Masahiro Tanaka,Andrew Cashner,NYY,BAL,...,1.98,182.0,4.171978,488768.0,4.68,1.35,6.48,3.48,150.0,4.516667
4,2019-03-28,Milwaukee Brewers,St. Louis Cardinals,5,4,1,Jhoulys Chacín,Miles Mikolas,MIL,STL,...,4.01,103.1,5.681523,571945.0,4.16,1.22,7.04,1.57,184.0,4.014130


In [8]:
# Class balance and missing-value check
# Note: rolling stat columns will show NaN for the first game of each season
# per team (no prior games to average) — this is expected; XGBoost handles NaN natively.
print("home_win distribution:")
print(training_df['home_win'].value_counts())
print()
nulls = training_df.isnull().sum()
print("Null counts:", nulls[nulls > 0].to_dict() or "none")

home_win distribution:
home_win
1    7949
0    7101
Name: count, dtype: int64

Null counts: {'home_starter_name': 7393, 'away_starter_name': 7402, 'home_rolling_runs_scored': 82, 'home_rolling_runs_allowed': 82, 'home_rolling_win_pct': 82, 'home_rolling_run_diff': 82, 'away_rolling_runs_scored': 86, 'away_rolling_runs_allowed': 86, 'away_rolling_win_pct': 86, 'away_rolling_run_diff': 86, 'home_pitcher_id': 7395, 'home_sp_era': 7395, 'home_sp_whip': 7395, 'home_sp_k_per_9': 7395, 'home_sp_bb_per_9': 7395, 'home_sp_ip': 7395, 'home_sp_fip_computed': 7395, 'away_pitcher_id': 7404, 'away_sp_era': 7404, 'away_sp_whip': 7404, 'away_sp_k_per_9': 7404, 'away_sp_bb_per_9': 7404, 'away_sp_ip': 7404, 'away_sp_fip_computed': 7404}


## 4c. Model Training

Train an XGBoost classifier on the training set, evaluate it, and persist the model and metrics.

In [ ]:
from datetime import date
from mlb_edge_finder import model

# train() does a 60/20/20 split: 60% fits XGBoost, 20% reserved for calibration (X_val),
# 20% held out for final evaluation (X_test).
clf, X_val, X_test, y_val, y_test = model.train(training_df)
print(f"Val set size:  {len(X_val)} games  (used for calibration)")
print(f"Test set size: {len(X_test)} games (used for evaluation)")
print(f"Features used: {list(X_test.columns)}")

In [ ]:
# Wrap the raw XGBoost model with isotonic probability calibration.
# calibrate() uses FrozenEstimator + CalibratedClassifierCV so XGBoost is NOT
# retrained — only the isotonic calibration layer is fit on the held-out val set.
cal_clf = model.calibrate(clf, X_val, y_val)

raw_metrics = model.evaluate(clf, X_test, y_test)
cal_metrics = model.evaluate(cal_clf, X_test, y_test)

import pandas as pd
comparison = pd.DataFrame(
    {"Raw XGBoost": raw_metrics, "Calibrated XGBoost": cal_metrics},
    index=raw_metrics.keys(),
)
print(comparison.to_string())
print("\nCalibrated model has lower Brier score → better probability estimates.")

In [10]:
baseline_clf, _, _ = model.train_baseline(training_df)
print("Baseline (logistic regression) trained.")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026-05-12 22:32:11,788 | INFO | mlb_edge_finder.model | Trained LogisticRegression baseline: 12040 samples


Baseline (logistic regression) trained.


In [ ]:
# Save the calibrated model. Pickle handles CalibratedClassifierCV transparently.
model.save_model(cal_clf, cal_metrics, date.today())

In [12]:
loaded_clf = model.load_model(date.today())
print("Model reloaded successfully.")
print(f"Sample predictions: {loaded_clf.predict(X_test[:3])}")

Model reloaded successfully.
Sample predictions: [0 0 1]


## 5. Find Edges

Run inference on today's features and flag games where the model finds positive expected value.

In [13]:
from mlb_edge_finder import edge_finder

edges = edge_finder.find_edges(features_df, clf, game_date)
print(f"{len(edges)} edge(s) found for {game_date}")
edges

2026-05-12 22:32:11,851 | WARNING | mlb_edge_finder.edge_finder | No features available for 2026-05-12 — returning empty edges


0 edge(s) found for 2026-05-12


,game_id,home_team,away_team,bet_side,american_odds,model_prob,ev,kelly_fraction


### 5b. Full Pipeline (end-to-end)

`pipeline.run()` drives all five stages — odds fetch, stats fetch, feature build, model load, edge find — in one call.

In [14]:
from mlb_edge_finder import pipeline

# Runs end-to-end for today: fetch odds, stats, build features, load model, find edges
pipeline_edges = pipeline.run(game_date)
pipeline_edges

2026-05-12 22:32:11,859 | INFO | mlb_edge_finder.pipeline | Running pipeline for 2026-05-12
2026-05-12 22:32:12,176 | INFO | mlb_edge_finder.features | Wrote 0 rows to /Users/jaydengould/Documents/projects/mlb-edge-finder/data/processed/features_2026-05-12.csv
2026-05-12 22:32:12,179 | INFO | mlb_edge_finder.pipeline | Loaded model from xgb_2026-05-12.pkl
2026-05-12 22:32:12,180 | WARNING | mlb_edge_finder.edge_finder | No features available for 2026-05-12 — returning empty edges


,game_id,home_team,away_team,bet_side,american_odds,model_prob,ev,kelly_fraction


## 7. Starting Pitcher Features

Fetch season-to-date individual pitcher stats via the MLB Stats API, and look up today's probable starters.

In [15]:
from mlb_edge_finder.pitcher_ingestion import (
    fetch_pitcher_stats,
    load_cached_pitcher_stats,
    fetch_probable_starters,
)
from datetime import date

# Fetch season pitcher stats (cached after first run)
# Columns: pitcher_id, pitcher_name, era, whip, k_per_9, bb_per_9, ip, fip_computed
snapshot_date = date(2025, 9, 28)
pitcher_df = fetch_pitcher_stats(snapshot_date)
print(f"Fetched {len(pitcher_df)} pitchers")
pitcher_df.head(10)


Fetched 873 pitchers


,pitcher_id,pitcher_name,era,whip,k_per_9,bb_per_9,ip,fip_computed
0,670183,Garrett Acton,0.0,2.00,0.00,18.00,1.0,9.150000
1,676070,Jacob Amaya,0.0,0.00,0.00,0.00,1.0,3.150000
2,682995,Hunter Barco,0.0,1.00,9.00,0.00,3.0,1.150000
3,595897,Nick Burdi,0.0,1.31,8.44,3.38,5.1,2.365686
4,676475,Alec Burleson,0.0,1.00,0.00,0.00,1.0,3.150000
5,621114,Ryan Burr,0.0,1.00,13.50,4.50,2.0,1.650000
6,650489,Willi Castro,0.0,1.00,0.00,0.00,1.0,3.150000
7,669722,Logan Davidson,0.0,0.00,9.00,0.00,1.0,1.150000
8,677649,Ezequiel Duran,0.0,0.30,0.00,0.00,3.1,3.150000
9,646242,Jhonathan Díaz,0.0,0.75,6.75,0.00,1.1,1.331818


In [16]:
# Fetch today's probable starters (live call, not cached — starters can change day-of)
today = date.today()
starters_df = fetch_probable_starters(today)
print(f"Probable starters for {today}: {len(starters_df)} games")
starters_df


Probable starters for 2026-05-12: 15 games


,home_abbr,away_abbr,home_starter_name,away_starter_name
0,CLE,LAA,Slade Cecconi,Walbert Ureña
1,BAL,NYY,Trevor Rogers,Will Warren
2,CIN,WSH,Brady Singer,Miles Mikolas
3,PIT,COL,Paul Skenes,Michael Lorenzen
4,BOS,PHI,Jovani Morán,Zack Wheeler
5,TOR,TB,Patrick Corbin,Shane McClanahan
6,NYM,DET,Freddy Peralta,Jack Flaherty
7,ATL,CHC,Grant Holmes,Colin Rea
8,CWS,KC,Erick Fedde,Stephen Kolek
9,MIN,MIA,Bailey Ober,Eury Pérez


In [17]:
# Verify pitcher sp columns appear in training set
sp_cols = [c for c in training_df.columns if c.startswith('home_sp_') or c.startswith('away_sp_')]
print(f"Pitcher feature columns: {sp_cols}")
training_df[['home_starter_name', 'away_starter_name'] + sp_cols[:4]].head()


Pitcher feature columns: ['home_sp_era', 'home_sp_whip', 'home_sp_k_per_9', 'home_sp_bb_per_9', 'home_sp_ip', 'home_sp_fip_computed', 'away_sp_era', 'away_sp_whip', 'away_sp_k_per_9', 'away_sp_bb_per_9', 'away_sp_ip', 'away_sp_fip_computed']


,home_starter_name,away_starter_name,home_sp_era,home_sp_whip,home_sp_k_per_9,home_sp_bb_per_9
0,Mike Fiers,Marco Gonzales,3.90,1.19,6.14,2.58
1,Marco Estrada,Yusei Kikuchi,6.85,1.31,4.18,3.04
2,Max Scherzer,Jacob deGrom,2.92,1.03,12.69,1.72
3,Masahiro Tanaka,Andrew Cashner,4.45,1.24,7.37,1.98
4,Jhoulys Chacín,Miles Mikolas,6.01,1.56,8.80,4.01


In [20]:
import pandas as pd

features = pd.read_csv("/Users/jaydengould/Documents/projects/mlb-edge-finder/data/processed/features_2026-05-18.csv")
cubs_row = features[
    (features["home_team"] == "Chicago Cubs") | 
    (features["away_team"] == "Chicago Cubs")
]
print(cubs_row.T)  # transpose so every feature prints on its own line

                                                          5
game_id                    a18a19981ef7d4e6c9f6244dc08e597a
home_team                                      Chicago Cubs
away_team                                 Milwaukee Brewers
home_odds_american                                     -150
away_odds_american                                      140
commence_time                          2026-05-18T23:41:00Z
home_abbr                                               CHC
away_abbr                                               MIL
home_bat_avg                                          0.246
home_obp                                              0.343
home_slg                                              0.406
home_ops                                              0.749
home_runs_per_game                                  5.12766
home_era                                               3.99
home_whip                                               1.2
home_k_per_9                            